In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        break

/kaggle/input/models/rambo09/softtg-toxcl-ihc/pytorch/default/2/tg_vocab.json
/kaggle/input/models/rambo09/softtg-toxcl-ihc/pytorch/default/2/best_ckpt/best_ckpt/config.json
/kaggle/input/datasets/rambo09/newmodel1-3/toxcl_soft_tg.py
/kaggle/input/datasets/veerhanglitch/toxcl-files/train.py
/kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_encoder_arch.py
/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv


In [2]:
INPUT_ROOT = "/kaggle/input/datasets"   # change only this if needed
INPUT_ROOT_PRAVEER = f"{INPUT_ROOT}/veerhanglitch"
INPUT_ROOT_RISHI = f"{INPUT_ROOT}/rambo09"

NEWMODEL_ROOT = f"{INPUT_ROOT_RISHI}/newmodel1-3"
DATA_ROOT = f"{INPUT_ROOT_PRAVEER}/toxcl-data"
WORK_ROOT = "/kaggle/working/toxcl_soft_tg"
OUTPUT_DIR = f"{WORK_ROOT}/outputs/SoftTG_ToXCL_IHC"

MODEL_NAME = "google/flan-t5-base"
TEACHER_NAME = "roberta-large"   # replace with fine-tuned teacher checkpoint if you have it
train_file = f"{WORK_ROOT}/train_soft_tg.py"

import os
import shutil

os.makedirs(WORK_ROOT, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

files_to_copy = [
    (f"{NEWMODEL_ROOT}/toxcl_soft_tg.py", f"{WORK_ROOT}/toxcl_soft_tg.py"),
    (f"{NEWMODEL_ROOT}/train_soft_tg.py", f"{WORK_ROOT}/train_soft_tg.py"),
]

for src, dst in files_to_copy:
    shutil.copy(src, dst)

print("Copied files to:", WORK_ROOT)
print(os.listdir(WORK_ROOT))

print(os.listdir("/kaggle/input"))
print(os.listdir(NEWMODEL_ROOT))
print(os.listdir(DATA_ROOT))

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

Copied files to: /kaggle/working/toxcl_soft_tg
['train_soft_tg.py', 'outputs', 'toxcl_soft_tg.py']
['models', 'datasets']
['toxcl_soft_tg.py', 'train_soft_tg.py']
['SBIC_valid.csv', 'SBIC_train.csv', 'IHC_valid.csv', 'SBIC_test.csv', 'IHC_train.csv', 'TG_valid.csv', 'TG_train.csv']


In [3]:
!pip install -q bert-score sacrebleu rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.1 MB/s eta 0:00:00


In [ ]:
%cd /kaggle/working/toxcl_soft_tg

!python train_soft_tg.py \
  --model_name_or_path {MODEL_NAME} \
  --teacher_name_or_path {TEACHER_NAME} \
  --dataset_name IHC \
  --data_root {DATA_ROOT} \
  --output_dir {OUTPUT_DIR} \
  --text_column_num 1 \
  --num_epochs 3 \
  --train_batch_size 8 \
  --valid_batch_size 16 \
  --learning_rate 2e-5 \
  --accumulation_steps 4 \
  --tg_top_k 3

In [5]:
# INFERENCE

import csv
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


MODEL_CODE_ROOT = f"{INPUT_ROOT_RISHI}/newmodel1-3"                                 # for toxcl_soft_tg.py is
INPUT_ROOT_RISHI_MODELS = "/kaggle/input/models/rambo09"
MODEL_ROOT = f"{INPUT_ROOT_RISHI_MODELS}/softtg-toxcl-ihc/pytorch/default/2"
CKPT_ROOT = f"{MODEL_ROOT}/best_ckpt/best_ckpt"  
# ^ replace with the Kaggle input path where you upload best_ckpt

VALID_FILE = f"{DATA_ROOT}/IHC_valid.csv"

NUM_SAMPLES = 15

DEVICE = torch.device("cpu")


import sys
if MODEL_CODE_ROOT not in sys.path:
    sys.path.append(MODEL_CODE_ROOT)

from toxcl_soft_tg import (
    ToXCL,
    extract_target_groups_from_augmented_text,
    build_target_group_vocab,
    encode_target_groups_batch,
)

# =========================
# LOAD CSV
# Assumes same column structure as your training:
# col 1 = augmented text
# col 2 = label
# col 4 = explanation
# =========================
def load_data(file_path, text_column_num=1):
    data = []
    with open(file_path, encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)
        for row in reader:
            data.append({
                "document": row[text_column_num].strip(),
                "label": row[2].strip(),
                "summary": row[4].strip(),
            })
    return data

valid_data = load_data(VALID_FILE, text_column_num=1)

# load/build vocab
tg_vocab_json = os.path.join(MODEL_ROOT, "tg_vocab.json")

if os.path.exists(tg_vocab_json):
    with open(tg_vocab_json, "r", encoding="utf-8") as f:
        tg_vocab = json.load(f)
else:
    # fallback: rebuild from validation data
    tg_vocab = build_target_group_vocab(valid_data)

print("tg_vocab_json path:", tg_vocab_json)
print("exists?", os.path.exists(tg_vocab_json))
print("tg vocab size:", len(tg_vocab))
print("num_target_groups passed to model:", len(tg_vocab) + 1)

tokenizer = AutoTokenizer.from_pretrained(CKPT_ROOT)
if tokenizer.pad_token is None:
    if tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})

decoder_model = AutoModelForSeq2SeqLM.from_pretrained(CKPT_ROOT).to(DEVICE)

model = ToXCL(
    decoder_model=decoder_model,
    num_target_groups=len(tg_vocab) + 1,
    pad_token_id=tokenizer.pad_token_id,
).to(DEVICE)

ckpt = torch.load(os.path.join(CKPT_ROOT, "soft_tg_ckpt.pt"), map_location="cpu")
state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
model.load_state_dict(state_dict, strict=False)
model.eval()

def predict_one(text, top_k=3, max_length=256, max_new_tokens=50):
    # tokenize augmented input
    enc = tokenizer(
        [text],
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    # parse TGs from augmented text
    parsed_tgs = extract_target_groups_from_augmented_text(text)

    tg_ids, tg_mask = encode_target_groups_batch([parsed_tgs], tg_vocab, top_k=top_k)
    tg_ids = tg_ids.to(DEVICE)
    tg_mask = tg_mask.to(DEVICE)

    with torch.no_grad():
        # dummy labels just to satisfy forward signature
        dummy_lm_labels = torch.full_like(input_ids, -100)
        dummy_cls_labels = torch.zeros((1,), dtype=torch.long, device=DEVICE)

        cls_logits, _, _, _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            lm_labels=dummy_lm_labels,
            cls_labels=dummy_cls_labels,
            tg_ids=tg_ids,
            tg_mask=tg_mask,
            teacher_logits=None,
        )

        pred_label = int(torch.argmax(cls_logits, dim=-1).item())

        gen_ids = model.generate_expl(
            input_ids=input_ids,
            attention_mask=attention_mask,
            tg_ids=tg_ids,
            tg_mask=tg_mask,
            num_beams=4,
            max_new_tokens=max_new_tokens,
        )
        gen_text = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

        if pred_label == 0:
            gen_text = "None"

    return parsed_tgs, pred_label, gen_text

for i in range(min(NUM_SAMPLES, len(valid_data))):
    sample = valid_data[i]
    parsed_tgs, pred_label, pred_expl = predict_one(sample["document"])

    print("=" * 120)
    print(f"Sample {i}")
    print("INPUT:")
    print(sample["document"])
    print()
    print("PARSED TARGET GROUPS:")
    print(parsed_tgs)
    print()
    print("PREDICTED LABEL:")
    print(pred_label, "(1 = toxic, 0 = non-toxic)")
    print()
    print("GENERATED EXPLANATION:")
    print(pred_expl)
    print()
    print("GOLD LABEL:")
    print(sample["label"])
    print()
    print("GOLD EXPLANATION:")
    print(sample["summary"])
    print()

tg_vocab_json path: /kaggle/input/models/rambo09/softtg-toxcl-ihc/pytorch/default/2/tg_vocab.json
exists? True
tg vocab size: 23
num_target_groups passed to model: 24


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Sample 0
INPUT:
Target: none Post: people like you are directly responsible for this .

PARSED TARGET GROUPS:
[]

PREDICTED LABEL:
0 (1 = toxic, 0 = non-toxic)

GENERATED EXPLANATION:
None

GOLD LABEL:
normal

GOLD EXPLANATION:
normal <SEP> none

Sample 1
INPUT:
Target: none Post: " conservatism " " in the us means libertarianism only  which goes even further than classical liberalism "

PARSED TARGET GROUPS:
[]

PREDICTED LABEL:
0 (1 = toxic, 0 = non-toxic)

GENERATED EXPLANATION:
None

GOLD LABEL:
normal

GOLD EXPLANATION:
normal <SEP> none

Sample 2
INPUT:
Target: none Post: pl get lost or i will block you

PARSED TARGET GROUPS:
[]

PREDICTED LABEL:
0 (1 = toxic, 0 = non-toxic)

GENERATED EXPLANATION:
None

GOLD LABEL:
normal

GOLD EXPLANATION:
normal <SEP> none

Sample 3
INPUT:
Target: none Post: fuhrer was madly loved by his people .

PARSED TARGET GROUPS:
[]

PREDICTED LABEL:
0 (1 = toxic, 0 = non-toxic)

GENERATED EXPLANATION:
None

GOLD LABEL:
normal

GOLD EXPLANATION:
normal <